# Voice Cloning Training with Coqui TTS (XTTS-v2)
This notebook uses **Coqui XTTS-v2** to create a voice clone from your Speech2Text app recordings.

## What is XTTS-v2?
XTTS-v2 is a state-of-the-art multilingual voice cloning model that can:
- Clone voices with just **6-10 seconds of audio**
- Work in **multiple languages** (German, English, Polish, etc.)
- Generate natural-sounding speech
- Fine-tune for better quality

### Two Approaches:

**Option 1: Zero-Shot Cloning (Fast - 5 minutes)**
- ✅ Uses 6-10 seconds of your voice
- ✅ Instant results
- ⚠️ Lower quality, may have artifacts

**Option 2: Fine-Tuning (Better - 1-2 hours)**
- ✅ Uses 10-20+ recordings
- ✅ Much better quality
- ✅ More natural sounding
- ⚠️ Takes longer

## Prerequisites:
- Exported training data from Speech2Text app to Google Drive
- At least 10-20 high-quality recordings (for fine-tuning)
- GPU runtime enabled in Colab

## Steps:
1. Setup GPU and mount Google Drive
2. Install Coqui TTS
3. Load your training data
4. Option A: Zero-shot voice cloning OR
5. Option B: Fine-tune for better quality
6. Test and export the model

## Step 1: Setup GPU and Mount Google Drive

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Please enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set the path to your training data (exported from Speech2Text app)
TRAINING_DATA_PATH = '/content/drive/MyDrive/TTS_Voice_Samples'

# Verify the data exists
if os.path.exists(TRAINING_DATA_PATH):
    wav_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.wav')]
    csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]
    
    print(f"\n✅ Found {len(wav_files)} WAV files and {len(csv_files)} CSV metadata file(s)")
    
    # Show CSV file details
    if csv_files:
        csv_file = os.path.join(TRAINING_DATA_PATH, csv_files[0])
        with open(csv_file, 'r', encoding='utf-8') as f:
            line_count = len(f.readlines())
        print(f"   CSV file: {csv_files[0]}")
        print(f"   Contains {line_count} training pairs")
        
        # Read language from CSV
        with open(csv_file, 'r', encoding='utf-8') as f:
            first_line = f.readline().strip()
            if '|' in first_line:
                parts = first_line.split('|')
                if len(parts) >= 3:
                    detected_lang = parts[2].strip()
                    print(f"   Detected language: {detected_lang}")
    else:
        print("⚠️ No CSV metadata file found!")
        print("   Make sure to export training data from the app using the 💾 button.")
else:
    print("\n❌ Training data folder not found. Please export data from the app first.")
    print(f"   Expected location: {TRAINING_DATA_PATH}")

## Step 2: Install Coqui TTS (XTTS-v2)

In [ ]:
# Install Coqui TTS (this may take 2-3 minutes)
print("📥 Installing Coqui TTS and dependencies...")
print("   This may take 2-3 minutes. Please wait...\n")

!pip install TTS
!pip install pydub
!pip install librosa soundfile

print("\n✅ Coqui TTS installation complete!")

In [ ]:
# Import and verify installation
try:
    from TTS.api import TTS
    import torch
    print("✅ TTS module imported successfully")
    
    # Check available models
    print("\n📦 Available XTTS models:")
    tts_models = TTS().list_models()
    xtts_models = [m for m in tts_models if 'xtts' in m.lower()]
    for model in xtts_models:
        print(f"   - {model}")
except ImportError as e:
    print(f"❌ Error importing TTS: {e}")
    print("\n⚠️ Please restart the runtime and try again:")
    print("   Runtime → Restart runtime")
    print("   Then run all cells again from the beginning.")

In [ ]:
# Download XTTS-v2 model (takes ~2 minutes)
print("📥 Downloading XTTS-v2 model...")
print("   This may take 2-3 minutes...\n")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda" if torch.cuda.is_available() else "cpu")

print("\n✅ XTTS-v2 model loaded successfully!")
print("   Supported languages: en, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, ja, hu, ko, hi")

## Step 3: Prepare Your Voice Samples

In [ ]:
import os
import shutil
import librosa
import soundfile as sf
from pydub import AudioSegment

# Read metadata from CSV
csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]

if not csv_files:
    print("❌ No CSV file found! Please export data from the app.")
else:
    csv_file = os.path.join(TRAINING_DATA_PATH, csv_files[0])
    
    # Parse CSV
    recordings = []
    with open(csv_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('|')
            if len(parts) >= 3:
                filename, text, lang = parts[0], parts[1], parts[2]
                recordings.append({
                    'file': filename,
                    'text': text,
                    'lang': lang
                })
    
    print(f"✅ Found {len(recordings)} recordings")
    
    # Map language codes to XTTS language codes
    lang_map = {
        '🇩🇪 Deutsch': 'de',
        '🇬🇧 English': 'en',
        '🇵🇱 Polski': 'pl'
    }
    
    if recordings:
        detected_lang = lang_map.get(recordings[0]['lang'], 'en')
        print(f"   Language: {recordings[0]['lang']} → {detected_lang}")
        
        # Show samples
        print(f"\n📝 Sample recordings:")
        for i, rec in enumerate(recordings[:3]):
            print(f"   {i+1}. {rec['file']}")
            print(f"      Text: {rec['text'][:60]}...")
        
        LANGUAGE = detected_lang

## Step 4A: Zero-Shot Voice Cloning (Quick Test)
Use one recording to clone your voice immediately - no training needed!

In [ ]:
from IPython.display import Audio, display
import random

# Select a random recording as speaker reference
if recordings:
    reference_rec = random.choice(recordings)
    reference_audio = os.path.join(TRAINING_DATA_PATH, reference_rec['file'])
    
    print(f"🎤 Using as voice reference: {reference_rec['file']}")
    print(f"   Text: {reference_rec['text']}")
    
    # Play original
    print("\n🔊 Original recording:")
    display(Audio(reference_audio))
    
    # Generate new speech with cloned voice
    test_text = "Dies ist ein Test meiner geklonten Stimme. Die Qualität kann mit Fine-Tuning noch verbessert werden."
    
    print(f"\n🤖 Generating speech with your cloned voice...")
    print(f"   Text: {test_text}")
    
    output_path = "/content/zero_shot_test.wav"
    
    tts.tts_to_file(
        text=test_text,
        speaker_wav=reference_audio,
        language=LANGUAGE,
        file_path=output_path
    )
    
    print("\n🔊 Generated speech with your voice:")
    display(Audio(output_path))
    
    print("\n💡 This is zero-shot cloning. For better quality, continue to Step 4B for fine-tuning!")
else:
    print("❌ No recordings found!")

## Step 4B: Fine-Tune XTTS-v2 with Your Voice (Better Quality)
This takes 1-2 hours but produces much better results!

In [ ]:
# Prepare dataset for fine-tuning
import json
from pathlib import Path

# Create output directories
FINETUNE_DIR = "/content/finetune_dataset"
os.makedirs(f"{FINETUNE_DIR}/wavs", exist_ok=True)

# Process and copy audio files
metadata = []
for rec in recordings:
    src = os.path.join(TRAINING_DATA_PATH, rec['file'])
    dst = os.path.join(f"{FINETUNE_DIR}/wavs", rec['file'])
    
    # Copy file
    shutil.copy2(src, dst)
    
    # Add to metadata
    metadata.append({
        "audio_file": f"wavs/{rec['file']}",
        "text": rec['text'],
        "speaker_name": "my_voice"
    })

# Save metadata as JSON
with open(f"{FINETUNE_DIR}/metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"✅ Prepared {len(metadata)} files for fine-tuning")
print(f"   Dataset location: {FINETUNE_DIR}")

In [ ]:
# Configure fine-tuning
OUTPUT_MODEL_DIR = "/content/finetuned_xtts"
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

# Fine-tuning configuration
config = {
    "language": LANGUAGE,
    "dataset_path": FINETUNE_DIR,
    "output_path": OUTPUT_MODEL_DIR,
    "num_epochs": 50,
    "batch_size": 2,
    "learning_rate": 1e-5,
}

print("⚙️ Fine-tuning configuration:")
for key, value in config.items():
    print(f"   {key}: {value}")

print("\n⏱️ Estimated time: 1-2 hours")
print("💡 You can monitor progress in the output below.")

In [ ]:
# Start fine-tuning (THIS TAKES 1-2 HOURS!)
print("🚀 Starting fine-tuning...\n")

!python -m TTS.bin.finetune \
    --model_name tts_models/multilingual/multi-dataset/xtts_v2 \
    --dataset_path {FINETUNE_DIR} \
    --output_path {OUTPUT_MODEL_DIR} \
    --language {LANGUAGE} \
    --num_epochs 50 \
    --batch_size 2 \
    --learning_rate 1e-5

print("\n\n✅ Fine-tuning complete!")
print(f"   Model saved to: {OUTPUT_MODEL_DIR}")

## Step 5: Test the Fine-Tuned Model

In [ ]:
from IPython.display import Audio, display

# Load the fine-tuned model (if you completed Step 4B)
# Otherwise this will use zero-shot cloning from Step 4A

try:
    # Try to load fine-tuned model
    finetuned_tts = TTS(model_path=f"{OUTPUT_MODEL_DIR}/best_model.pth", config_path=f"{OUTPUT_MODEL_DIR}/config.json").to("cuda" if torch.cuda.is_available() else "cpu")
    print("✅ Loaded fine-tuned model")
    use_finetuned = True
except:
    print("⚠️ Fine-tuned model not found, using zero-shot cloning")
    finetuned_tts = tts
    use_finetuned = False

# Test sentences in different languages
test_sentences = {
    'de': "Hallo! Dies ist meine geklonte Stimme. Die Qualität ist durch das Fine-Tuning deutlich besser geworden.",
    'en': "Hello! This is my cloned voice. The quality has improved significantly through fine-tuning.",
    'pl': "Cześć! To jest mój sklonowany głos. Jakość znacznie się poprawiła dzięki dostrojeniu."
}

test_text = test_sentences.get(LANGUAGE, test_sentences['en'])

print(f"\n🤖 Generating test audio...")
print(f"   Text: {test_text}")

output_path = "/content/finetuned_test.wav"

if use_finetuned:
    # Fine-tuned model doesn't need speaker_wav
    finetuned_tts.tts_to_file(
        text=test_text,
        file_path=output_path
    )
else:
    # Zero-shot needs reference audio
    reference_audio = os.path.join(TRAINING_DATA_PATH, recordings[0]['file'])
    finetuned_tts.tts_to_file(
        text=test_text,
        speaker_wav=reference_audio,
        language=LANGUAGE,
        file_path=output_path
    )

print("\n🔊 Generated audio:")
display(Audio(output_path))

## Step 6: Export Model to Google Drive

In [ ]:
import shutil
from datetime import datetime

# Create export directory
export_dir = '/content/drive/MyDrive/XTTS_Voice_Models'
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_export_dir = f"{export_dir}/my_voice_{LANGUAGE}_{timestamp}"
os.makedirs(model_export_dir, exist_ok=True)

if use_finetuned and os.path.exists(OUTPUT_MODEL_DIR):
    # Export fine-tuned model
    print("📦 Exporting fine-tuned model...")
    
    # Copy all model files
    for file in os.listdir(OUTPUT_MODEL_DIR):
        if file.endswith(('.pth', '.json', '.txt')):
            src = os.path.join(OUTPUT_MODEL_DIR, file)
            dst = os.path.join(model_export_dir, file)
            shutil.copy2(src, dst)
            print(f"   ✅ {file}")
    
    # Copy test audio
    if os.path.exists('/content/finetuned_test.wav'):
        shutil.copy2('/content/finetuned_test.wav', f"{model_export_dir}/test_sample.wav")
        print(f"   ✅ test_sample.wav")
    
    print(f"\n✅ Fine-tuned model exported to: {model_export_dir}")
else:
    # Export reference audio for zero-shot
    print("📦 Exporting reference audio for zero-shot cloning...")
    
    reference_audio = os.path.join(TRAINING_DATA_PATH, recordings[0]['file'])
    shutil.copy2(reference_audio, f"{model_export_dir}/speaker_reference.wav")
    
    # Save metadata
    with open(f"{model_export_dir}/info.txt", 'w') as f:
        f.write(f"Language: {LANGUAGE}\n")
        f.write(f"Model: XTTS-v2 Zero-Shot\n")
        f.write(f"Reference text: {recordings[0]['text']}\n")
    
    print(f"\n✅ Reference audio exported to: {model_export_dir}")
    print("   Use this with XTTS-v2 for zero-shot cloning.")

## Step 7: Create Usage Instructions

In [ ]:
# Create README with usage instructions
readme_content = f"""# Voice Clone Model - {LANGUAGE}

Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Language: {LANGUAGE}
Model: {"Fine-tuned XTTS-v2" if use_finetuned else "XTTS-v2 Zero-Shot"}

## Usage in Python:

```python
from TTS.api import TTS
import torch

# Load model
{'tts = TTS(model_path="best_model.pth", config_path="config.json")' if use_finetuned else 'tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")'}
tts = tts.to("cuda" if torch.cuda.is_available() else "cpu")

# Generate speech
{'tts.tts_to_file(text="Your text here", file_path="output.wav")' if use_finetuned else 'tts.tts_to_file(text="Your text here", speaker_wav="speaker_reference.wav", language="' + LANGUAGE + '", file_path="output.wav")'}
```

## Quality:

{"- Fine-tuned on your voice recordings" if use_finetuned else "- Zero-shot cloning using reference audio"}
- Best for {LANGUAGE} language
- Sample audio: test_sample.wav

## Integration with Android:

For Android integration, you'll need to:
1. Convert the model to ONNX format
2. Use ONNX Runtime on Android
3. Or use a server-based approach with this Python model

## Further Improvement:

To improve quality:
1. Record more diverse sentences (20-50 clips)
2. Ensure consistent audio quality
3. Run Step 4B again with more epochs
"""

with open(f"{model_export_dir}/README.md", 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("✅ Created README.md with usage instructions")
print(f"\n📁 Complete export location: {model_export_dir}")

## Next Steps:

1. **Download the trained model** (`.onnx` and `.onnx.json` files)
2. **Integrate with your Android app** using Piper TTS library
3. **Test the voice** in your app
4. **Continue improving**: Record more data and run Step 6b to refine the model

## Fine-Tuning Tips:

- **Start small**: 10-20 recordings are enough to start
- **Incremental improvement**: Add 5-10 more recordings and re-train
- **Consistent quality**: Same environment, same microphone
- **Clear pronunciation**: Speak naturally and clearly
- **Vary your sentences**: Use different sentence structures
- **Monitor quality**: Test after each training session

## Why Fine-Tuning is Better:

- ✅ **Faster Results**: Get a working model in 30 minutes instead of 10+ hours
- ✅ **Less Data**: 10-20 samples work well, not 100+
- ✅ **Better Quality**: Pre-trained model already knows language patterns
- ✅ **Iterative**: Keep improving by adding more data
- ✅ **Cost Efficient**: Less GPU time needed on Colab

## Troubleshooting:

- **If quality is poor after first training**: Add 10 more recordings and re-train
- **If voice sounds robotic**: Increase recording length (3-8 seconds per clip)
- **If training fails**: Check that all WAV files have matching TXT files
- **To improve specific words**: Record more sentences with those words